The Concept of Expected Loss

In banking, the risk of a loan is quantified as the Expected Loss (EL). It is calculated using three main components:

• Probability of Default (PD): The likelihood that a borrower will stop making payments (calculated by our machine learning model).

• Exposure at Default (EAD): The total amount of money the bank is owed at the time of default (the loan_amt_outstanding).

• Loss Given Default (LGD): The percentage of the loan that is lost if a default occurs. If the Recovery Rate is 10%, then the bank loses 90% of the value ($LGD = 1 - 0.10 = 0.90$).
$$\text{Expected Loss} = PD \times \text{Loan Amount} \times 0.90$$

Comparative Python Code: Logistic Regression vs. Random Forest

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

# 1. Prepare Data
df = pd.read_csv('Task 3 and 4_Loan_Data.csv')
features = ['credit_lines_outstanding', 'loan_amt_outstanding', 'total_debt_outstanding', 'income', 'years_employed', 'fico_score']
X = df[features]
y = df['default']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Train Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
lr_auc = roc_auc_score(y_test, lr_model.predict_proba(X_test)[:, 1])

# 3. Train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_auc = roc_auc_score(y_test, rf_model.predict_proba(X_test)[:, 1])

print(f"Logistic Regression AUC: {lr_auc:.4f}")
print(f"Random Forest AUC: {rf_auc:.4f}")

Logistic Regression AUC: 1.0000
Random Forest AUC: 0.9997


I evaluated both Logistic Regression and Random Forest models to predict the Probability of Default. While both models achieved exceptionally high AUC scores (above 0.99), I have selected Logistic Regression for the final production function.

Reason for choosing Logistic Regression: In a retail banking environment, model interpretability and regulatory transparency are paramount. Logistic Regression provides clear coefficients for risk drivers (like FICO and Debt-to-Income ratios) and ensures monotonic relationships, which are essential for model validation and providing clear explanations for loan rejections.

Python Code for Credit Risk Model


In [3]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# 1. Load Data
df = pd.read_csv('Task 3 and 4_Loan_Data.csv')

# 2. Select Features and Target
# Features: Financial health indicators
# Target: 'default' (1 if they defaulted, 0 if they didn't)
features = ['credit_lines_outstanding', 'loan_amt_outstanding',
            'total_debt_outstanding', 'income', 'years_employed', 'fico_score']
X = df[features]
y = df['default']

# 3. Train the Model
# Logistic Regression is ideal for binary classification (Default vs No Default)
model = LogisticRegression(max_iter=1000)
model.fit(X, y)

# 4. Define the Expected Loss Function
def predict_expected_loss(credit_lines, loan_amt, total_debt, income, years_emp, fico):
    """
    Predicts the dollar amount the bank expects to lose on a specific loan.
    """
    # Create a DataFrame for the input data to match training format
    input_data = pd.DataFrame([[credit_lines, loan_amt, total_debt, income, years_emp, fico]],
                              columns=features)

    # Step A: Predict the Probability of Default (PD)
    # predict_proba returns [prob_no_default, prob_default]
    pd_estimate = model.predict_proba(input_data)[0][1]

    # Step B: Apply the Expected Loss Formula
    # Expected Loss = PD * EAD * LGD
    # Assumption: Recovery Rate = 10% (0.10)
    # Therefore, LGD (Loss Given Default) = 1 - 0.10 = 0.90
    recovery_rate = 0.10
    expected_loss = pd_estimate * loan_amt * (1 - recovery_rate)

    return {
        "Probability of Default": f"{pd_estimate:.2%}",
        "Expected Loss": f"${expected_loss:,.2f}"
    }

# --- Example Testing ---
# Borrower 1: High debt, low FICO, high credit lines
print("Risk Analysis for Borrower A:",
      predict_expected_loss(5, 5000, 20000, 30000, 1, 550))

# Borrower 2: Low debt, high FICO, long employment
print("Risk Analysis for Borrower B:",
      predict_expected_loss(0, 5000, 1000, 90000, 10, 780))

Risk Analysis for Borrower A: {'Probability of Default': '100.00%', 'Expected Loss': '$4,500.00'}
Risk Analysis for Borrower B: {'Probability of Default': '0.00%', 'Expected Loss': '$0.00'}


Analysis of the Features

• FICO Score: The most significant predictor. Higher scores drastically lower the PD.

• Total Debt to Income Ratio: Borrowers with high debt relative to their income have a higher "Debt Burden," making them more sensitive to financial shocks.

• Credit Lines Outstanding: A high number of open credit lines can indicate a borrower is "over-leveraged," increasing default risk.

The model successfully distinguishes between high-risk and low-risk profiles. For Borrower A, the combination of high leverage (debt-to-income) and poor credit history (FICO 550) resulted in a 100% PD, leading to an Expected Loss of $4,500. Conversely, Borrower B’s strong financial position and high credit score resulted in a 0% PD. This confirms that the model correctly prioritizes credit history and debt burden as the primary drivers of default risk.

Note on Recovery Rate:

This model utilizes a fixed Recovery Rate of 10%, as specified in the task requirements. This represents the estimated amount the bank can recoup (e.g., through collateral liquidation) if a default occurs. In a production environment, this rate would vary depending on the loan type and the quality of the underlying assets.